# Deploy Kimi K3 on Amazon SageMaker AI

[Kimi K3](https://huggingface.co/moonshotai/Kimi-K3) is Moonshot AI's open-weight, native multimodal agentic model — a 2.8T-parameter Mixture-of-Experts built on Kimi Delta Attention (KDA) and Attention Residuals (AttnRes), with a 1M-token context window. It is the first open model in the 3T-parameter class.

## Model summary

| Property | Value |
| :--- | :--- |
| Architecture | Mixture-of-Experts (MoE) |
| Total / activated parameters | 2.8T / 104B |
| Layers | 93 (69 KDA + 24 Gated MLA, 1 dense) |
| Experts | 896 total, 16 selected per token, 2 shared |
| Context length | 1,048,576 |
| Vision encoder | MoonViT-V2 (401M) |
| Quantization | MXFP4 weights / MXFP8 activations (quantization-aware training) |
| Modality | Text, image |
| License | [Kimi K3 License](https://huggingface.co/moonshotai/Kimi-K3/blob/main/LICENSE) |

Architecture and benchmark figures are quoted from the published Hugging Face model card. See also the [Kimi K3 tech blog](https://www.kimi.com/blog/kimi-k3).

## What is different from the Kimi K2.5 example

If you are coming from [`01-models/Kimi/LMI/kimi-k2.5.ipynb`](../LMI/kimi-k2.5.ipynb), three things change materially:

1. **Instance sizing.** K2.5 fits on one `ml.p5e.48xlarge` (8x H200, 1,128 GB HBM). K3's MXFP4 checkpoint is roughly 1.5 TB on disk, so the weights alone exceed that. The smallest single-instance SageMaker real-time option is `ml.p6-b200.48xlarge` (8x B200, 1,440 GB HBM), and headroom for KV cache is thin — hence the conservative `max_model_len` below.
2. **Parsers.** K3 uses `kimi_k3` for both the reasoning parser and the tool-call parser, not `kimi_k2`.
3. **Preserved thinking history.** K3 always runs with thinking enabled. On multi-turn and tool-calling requests you must echo the **complete** assistant message back into `messages` — including `reasoning_content` and `tool_calls`, not just `content`. There is a dedicated section on this below.


## Prerequisites

- A SageMaker execution role with permission to create models, endpoint configs, and endpoints.
- Service quota for your chosen instance type on **SageMaker inference** (this is a separate quota from EC2 and from SageMaker training).
- A Hugging Face token with access to `moonshotai/Kimi-K3`, if you are pulling weights from the Hub at container start.

### Instance sizing

| Instance | GPUs | Total HBM | Fits K3 (MXFP4, ~1.4 TB weights)? |
| :--- | :--- | :--- | :--- |
| `ml.p5e.48xlarge` | 8x H200 (141 GB) | 1,128 GB | No — weights alone exceed HBM |
| `ml.p5en.48xlarge` | 8x H200 (141 GB) | 1,128 GB | No |
| `ml.p6-b200.48xlarge` | 8x B200 (180 GB) | 1,440 GB | Yes, with reduced context |

> **On multi-node.** vLLM's published K3 recipe recommends at least 8x GB300 and multi-node for real production traffic. SageMaker real-time endpoints do not shard a single model across instances — `InitialInstanceCount > 1` creates independent replicas, each of which must hold the full model. If one instance cannot hold K3 at your target context length, the deployment target is [SageMaker HyperPod](../../../SageMakerHyperpod/) rather than a real-time endpoint.

### Weight loading

Pulling ~1.5 TB from the Hugging Face Hub at container start is slow and is the most common cause of a failed health check on this model. Two mitigations, both used below:

- `ContainerStartupHealthCheckTimeoutInSeconds` is set to the 3600s maximum.
- For repeat deployments, stage the weights in S3 once and point the container at them. The vLLM DLC bundles `runai-model-streamer`, which streams weights directly from S3 and is substantially faster than a Hub download.


In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import base64
import json
import re
import sys
import time

import boto3
from IPython.display import display, Image, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")            # control plane
sm_runtime = boto3.client("sagemaker-runtime")  # data plane

print(f"region: {region}")

In [ ]:
#
# Helper functions — no SageMaker Python SDK dependency
#
def get_sagemaker_role():
    arn = boto3.client("sts").get_caller_identity()["Arn"]
    return re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", arn)


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 60):
    """Poll an endpoint until it leaves the Creating/Updating state."""
    progress = ""
    while True:
        desc = sm.describe_endpoint(EndpointName=endpoint_name)
        status = desc["EndpointStatus"]
        if status not in ("Creating", "Updating"):
            break
        progress += "."
        clear_output(wait=True)
        print(f"Waiting for '{endpoint_name}': {progress}")
        time.sleep(sleep_time)
    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")
    if status == "Failed":
        print(desc.get("FailureReason", "(no failure reason returned)"))
    return status

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside SageMaker Studio
#
role = None

if role is None:
    role = get_sagemaker_role()
print(role)

## Configuration

In [ ]:
instance = {"type": "ml.p6-b200.48xlarge", "num_gpu": 8}
model_id = "moonshotai/Kimi-K3"

model_name = f"kimi-k3-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
variant_name = "v1"

# Maximum allowed by SageMaker. K3 needs it: the checkpoint is ~1.5 TB.
startup_timeout = 3600

# Well below the model's 1M-token ceiling. On a single 8x B200 node the weights
# consume most of HBM, so KV cache budget is the binding constraint. Raise this
# only after confirming the endpoint comes up and you have measured free memory.
max_model_len = 131072

hf_token = "<YOUR_HF_TOKEN>"  # needs access to moonshotai/Kimi-K3

## Deployment options

Pick **one** of the two cells below and run only that one. Both containers expose an OpenAI-compatible API on port 8080, so every inference cell in this notebook works unchanged against either.

> **Container version.** K3's KDA attention, Attention Residuals, and MXFP4 MoE path are new model code. Support landed in vLLM and SGLang alongside the weight release, so you need a DLC tag built after that. Check the [vLLM DLC](https://aws.github.io/deep-learning-containers/vllm/) and [SGLang DLC](https://aws.github.io/deep-learning-containers/sglang/) release pages and pin the first tag that lists Kimi K3 / KDA support. The tags below are placeholders — update them before running.


### Option 1: vLLM

Serves K3 with the [AWS Deep Learning Container for vLLM](https://aws.github.io/deep-learning-containers/vllm/). The image bundles FlashInfer (which supplies the MXFP4 MoE runner on Blackwell), DeepEP for expert-parallel kernels, `runai-model-streamer` for S3 weight streaming, and EFA/OpenMPI.

Flag notes:

- `SM_VLLM_TOOL_CALL_PARSER=kimi_k3` — required for tool calling.
- `SM_VLLM_REASONING_PARSER=kimi_k3` — required. K3 always emits reasoning; without this the thinking text is not split out into `reasoning_content`.
- `SM_VLLM_MM_ENCODER_TP_MODE=data` — runs the small MoonViT-V2 encoder data-parallel instead of tensor-parallel, avoiding TP communication overhead on a 401M module.
- `SM_VLLM_TRUST_REMOTE_CODE=true` — K3 ships custom modeling code (`configuration_kimi_k3.py`, `encoding_k3.py`).


In [ ]:
CONTAINER_TAG = "0.24.0-gpu-py312-cu130-ubuntu22.04-sagemaker"  # verify K3 support before running
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:{CONTAINER_TAG}"

common_env = {
    "HF_TOKEN": hf_token,
    "SM_NUM_GPUS": json.dumps(instance["num_gpu"]),
}

vllm_env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": str(max_model_len),
    "SM_VLLM_TRUST_REMOTE_CODE": "true",
    "SM_VLLM_MM_ENCODER_TP_MODE": "data",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_TOOL_CALL_PARSER": "kimi_k3",
    "SM_VLLM_REASONING_PARSER": "kimi_k3",
    # KV cache in FP8 roughly halves KV memory — meaningful headroom on a single node.
    "SM_VLLM_KV_CACHE_DTYPE": "fp8",
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.92",
    # Chunked prefill keeps peak activation memory down during long-context prefill.
    "SM_VLLM_MAX_NUM_BATCHED_TOKENS": "32768",
}
env = common_env | vllm_env

reasoning_keyword = "reasoning_content"

### Option 2: SGLang

Serves K3 with the [AWS Deep Learning Container for SGLang](https://aws.github.io/deep-learning-containers/sglang/). SGLang's K3 recipes expose knobs specific to the KDA state cache, which is the part of K3 that classic paged-KV tuning does not cover.

Flag notes:

- `SM_SGLANG_MAMBA_SSM_DTYPE=bfloat16` — KDA recurrent state defaults to FP32; BF16 halves state memory.
- `SM_SGLANG_MAMBA_RADIX_CACHE_STRATEGY=extra_buffer_lazy` — prefix-cache strategy adapted to KDA, which cannot use conventional radix caching unmodified.
- `SM_SGLANG_KV_CACHE_DTYPE=fp8_e4m3` — halves KV memory for the 24 MLA layers.
- `SM_SGLANG_DISABLE_CUSTOM_ALL_REDUCE` + `SM_SGLANG_ENABLE_SYMM_MEM` — follows the published Blackwell single-node recipe.


In [ ]:
CONTAINER_TAG = "0.6.0-gpu-py312-cu130-ubuntu24.04-sagemaker"  # verify K3 support before running
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/sglang:{CONTAINER_TAG}"

common_env = {
    "HF_TOKEN": hf_token,
}

sgl_env = {
    "SM_SGLANG_MODEL_PATH": model_id,
    "SM_SGLANG_TP": json.dumps(instance["num_gpu"]),
    "SM_SGLANG_CONTEXT_LENGTH": str(max_model_len),
    "SM_SGLANG_TRUST_REMOTE_CODE": "true",
    "SM_SGLANG_TOOL_CALL_PARSER": "kimi_k3",
    "SM_SGLANG_REASONING_PARSER": "kimi_k3",
    "SM_SGLANG_KV_CACHE_DTYPE": "fp8_e4m3",
    "SM_SGLANG_MAMBA_SSM_DTYPE": "bfloat16",
    "SM_SGLANG_MAMBA_RADIX_CACHE_STRATEGY": "extra_buffer_lazy",
    "SM_SGLANG_MEM_FRACTION_STATIC": "0.85",
    "SM_SGLANG_DISABLE_CUSTOM_ALL_REDUCE": "true",
    "SM_SGLANG_ENABLE_SYMM_MEM": "true",
}
env = common_env | sgl_env

reasoning_keyword = "reasoning_content"

## Deployment

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)
print(f"Created model: {model_name}")

In [ ]:
capacity_reservation = {
    "CapacityReservationPreference": "capacity-reservations-only",
    "MlReservationArn": "<YOUR_CAPACITY_RESERVATION>",
}

_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": startup_timeout,
            "ModelDataDownloadTimeoutInSeconds": startup_timeout,
            "InferenceAmiVersion": "al2023-ami-sagemaker-inference-gpu-4-1",
            # Blackwell instances are scarce. Uncomment if you hold a reservation.
            # "CapacityReservationConfig": capacity_reservation,
        },
    ],
)

_ = sm.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)

wait_for_endpoint(endpoint_name)

## Inference

A small helper so the examples below stay readable. Everything speaks the OpenAI Chat Completions schema.


In [ ]:
def invoke(payload, show=True):
    """POST a Chat Completions payload to the endpoint and return the assistant message."""
    start = time.time()
    res = sm_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        Body=json.dumps(payload),
        ContentType="application/json",
    )
    response = json.loads(res["Body"].read().decode("utf8"))
    elapsed = time.time() - start

    msg = response["choices"][0]["message"]
    if show:
        print(f"Response time: {elapsed:.2f}s | usage: {response.get('usage')}\n")
        if msg.get(reasoning_keyword):
            display(Markdown("### Reasoning\n---"))
            display(Markdown(msg[reasoning_keyword]))
        if msg.get("content"):
            display(Markdown("### Content\n---"))
            display(Markdown(msg["content"]))
        sys.stdout.flush()
    return msg

### Text generation and thinking effort

K3 always thinks. Depth is controlled by the top-level `reasoning_effort` field, which accepts `"low"`, `"high"`, or `"max"` (default `"max"`). There is no way to turn thinking off — if you want short answers, turn the effort down rather than looking for an `enable_thinking` switch.


In [ ]:
msg = invoke({
    "messages": [
        {"role": "user", "content": "What model are you, and what is your context window?"}
    ],
    "reasoning_effort": "low",
})

In [ ]:
msg = invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "A service has 896 experts and routes 16 per token. Compare the "
                "all-reduce traffic of expert parallelism against tensor parallelism "
                "for this sparsity, and say which you would pick for decode."
            ),
        }
    ],
    "reasoning_effort": "max",
})

### Preserved thinking history

This is the part that most often breaks when porting K2-era code.

K3 was trained in preserved-thinking mode. On every follow-up request you must pass the **entire** assistant message back into `messages` exactly as the API returned it — `reasoning_content` included. If you append only `content`, the model loses access to its own prior reasoning and the second turn degrades in ways that are easy to misread as a quality problem with the model.

The cell below demonstrates the contract: the model is asked for three numbers while privately holding five, then asked for the other two. It can only answer if the reasoning came back with the history.


In [ ]:
messages = [{"role": "user", "content": "Tell me three random numbers."}]

first = invoke({"messages": messages, "reasoning_effort": "high"})

# Append the COMPLETE assistant message — not just first["content"].
messages.append(first)
messages.append({"role": "user", "content": "What are the other two numbers you had in mind?"})

second = invoke({"messages": messages, "reasoning_effort": "high"})

### Vision input

K3 is natively multimodal — text and images go through the same model, served by the MoonViT-V2 encoder. Images are passed as `image_url` content parts, either as a URL or as a base64 data URI.


In [ ]:
def image_data_uri(path: str) -> str:
    ext = path.rsplit(".", 1)[-1].lower()
    mime = {"jpg": "jpeg", "jpeg": "jpeg", "png": "png", "gif": "gif", "webp": "webp"}.get(ext, "jpeg")
    with open(path, "rb") as f:
        return f"data:image/{mime};base64,{base64.b64encode(f.read()).decode()}"


# Reuses the sample image from the Kimi K2.5 example in this repo.
image_path = "../LMI/schedule_table_small.jpg"
display(Image(filename=image_path))

msg = invoke({
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Transcribe this schedule table as CSV. Output only the CSV."},
                {"type": "image_url", "image_url": {"url": image_data_uri(image_path)}},
            ],
        }
    ],
    "reasoning_effort": "high",
})

### Agentic tool calling

K3's headline capability is long-horizon agentic work, so the tool loop matters more here than the single-shot examples above.

Two rules the loop below follows:

1. Append the complete assistant message (with `reasoning_content` **and** `tool_calls`) before appending any `tool` results.
2. Send the full accumulated `messages` list on every iteration.


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files in a directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Directory path"}
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write text content to a file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "File path to write"},
                    "content": {"type": "string", "description": "Content to write"},
                },
                "required": ["path", "content"],
            },
        },
    },
]


def list_files(path: str):
    import os
    print(f"  [tool] list_files({path})")
    try:
        return json.dumps(sorted(os.listdir(path)))
    except OSError as e:
        return f"Error: {e}"


def write_file(path: str, content: str):
    print(f"  [tool] write_file({path}, {len(content)} chars)")
    with open(path, "w") as f:
        f.write(content)
    return f"Wrote {len(content)} characters to {path}"


tool_functions = {"list_files": list_files, "write_file": write_file}

In [ ]:
def agent(user_input, max_turns=10):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an assistant with filesystem tools. Use the tools to do the "
                "work rather than describing what you would do. Keep going until the "
                "task is complete."
            ),
        },
        {"role": "user", "content": user_input},
    ]

    for turn in range(1, max_turns + 1):
        print(f"\n--- turn {turn} ---")
        msg = invoke(
            {
                "messages": messages,
                "tools": tools,
                "tool_choice": "auto",
                "reasoning_effort": "high",
            },
            show=False,
        )

        # The complete message goes back, reasoning_content and tool_calls included.
        messages.append(msg)

        if not msg.get("tool_calls"):
            display(Markdown(msg.get("content", "")))
            return msg.get("content")

        for tc in msg["tool_calls"]:
            fn = tc["function"]
            args = json.loads(fn["arguments"])
            result = tool_functions[fn["name"]](**args)
            messages.append(
                {"role": "tool", "tool_call_id": tc["id"], "content": str(result)}
            )

    print("Hit max_turns without a final answer.")
    return None


agent("List the files in the current directory, then write a file called inventory.md containing a markdown table of what you found.")

### OpenAI-compatible invocation

SageMaker endpoints expose an `/openai/v1` path, so the OpenAI SDK, LangChain, or Strands can talk to this endpoint by changing only the base URL. Useful if you are dropping K3 into an existing agent framework rather than calling `invoke_endpoint` directly.


In [ ]:
# %pip install --quiet openai
#
# from openai import OpenAI
# import boto3
#
# client = OpenAI(
#     base_url=f"https://runtime.sagemaker.{region}.amazonaws.com/endpoints/{endpoint_name}/openai/v1",
#     api_key="not-used",  # requests are signed with SigV4 by the SageMaker endpoint layer
# )
#
# resp = client.chat.completions.create(
#     model=endpoint_name,
#     messages=[{"role": "user", "content": "Hello"}],
#     reasoning_effort="high",
# )
# print(resp.choices[0].message.content)

print("See https://aws.amazon.com/blogs/machine-learning/announcing-openai-compatible-api-support-for-amazon-sagemaker-ai-endpoints/")

## Cleanup

`ml.p6-b200.48xlarge` is expensive. Run this as soon as you are finished.


In [ ]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)
print("Deleted endpoint, endpoint config, and model.")